
# ALDI Web Scraper

This notebook collects product and price information from ALDI using Selenium.

### How it works

1. Searches for the 20 product terms defined in `SEARCH_TERMS`.
2. Loads additional products through pagination and automatic retries.
3. Extracts product IDs, names, brands, selling prices, quantities, unit-price information, URLs, and timestamps.
4. Combines the collected results into a RAW dataset.
5. Validates collection completeness and saves RAW files.

### Data Storage

- Current RAW: `data/raw/aldi_products_raw.csv`
- Historical RAW: `data/history/aldi/YYYY-MM-DD_raw.csv`


Deduplication, quantity normalization, unit-price calculation, classification, and historical price processing are handled by the central pipeline.

### Collection Policy

- Complete collections update the current RAW and create a historical snapshot.
- Partial collections are archived without replacing the current RAW.
- Existing historical snapshots are never overwritten.
- TEST_MODE allows testing selected searches without saving files.


In [31]:
import time
import re
from pathlib import Path
from datetime import datetime

import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    ElementClickInterceptedException,
    TimeoutException
)

HOME_URL = "https://www.aldi.es/"
WAIT = 15
MAX_RETRIES = 2

SEARCH_TERMS = [
    "leche", "huevos", "manzana", "plátano", "tomate", "arroz",
    "pasta", "pan", "pollo", "carne picada", "salmón", "yogur",
    "queso", "agua", "zumo", "atún", "garbanzos", "aceite",
    "detergente", "papel higiénico"
]

# --------------------------------------------------
# TEST MODE
# --------------------------------------------------

TEST_MODE = True
TEST_TERMS = ["leche"]

TERMS_TO_SCRAPE = TEST_TERMS if TEST_MODE else SEARCH_TERMS

print("Search terms:", TERMS_TO_SCRAPE)

Search terms: ['leche']


In [32]:
# --------------------------------------------------
# PROJECT DIRECTORIES
# --------------------------------------------------

def find_project_root():
    current = Path.cwd().resolve()

    for path in (current, *current.parents):
        if path.name == "supermarket_project":
            return path

    if current.name == "scrapers":
        return current.parent

    raise FileNotFoundError("Could not locate the project root.")


BASE_DIR = find_project_root()

RAW_DIR = BASE_DIR / "data" / "raw"
HISTORY_DIR = BASE_DIR / "data" / "history" / "aldi"

RAW_PATH = RAW_DIR / "aldi_products_raw.csv"

RUN_DATE = datetime.now().strftime("%Y-%m-%d")
RAW_HISTORY_PATH = HISTORY_DIR / f"{RUN_DATE}_raw.csv"

print("Project:", BASE_DIR)
print("Collection date:", RUN_DATE)
print("Current RAW:", RAW_PATH)
print("Historical RAW:", RAW_HISTORY_PATH)

Project: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project
Collection date: 2026-11-13
Current RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\aldi_products_raw.csv
Historical RAW: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\history\aldi\2026-11-13_raw.csv


In [33]:
def start_driver():
    options = Options()
    options.add_argument("--window-size=1920,1080")
    driver = webdriver.Chrome(options=options)
    driver.get(HOME_URL)
    return driver


def accept_cookies(driver):
    container = driver.execute_script("""
        function find(selector, root = document) {
            const el = root.querySelector(selector);
            if (el) return el;
            for (const node of root.querySelectorAll('*')) {
                if (node.shadowRoot) {
                    const found = find(selector, node.shadowRoot);
                    if (found) return found;
                }
            }
            return null;
        }
        return find('[data-testid="uc-container"]');
    """)

    if container is None:
        print("No apareció el popup de cookies.")
        return

    scroller = driver.execute_script("""
        return arguments[0].getRootNode()
            .querySelector('[data-testid="uc-vertical-scroller"]');
    """, container)

    if scroller is not None:
        driver.execute_script(
            "arguments[0].scrollTop = arguments[0].scrollHeight;",
            scroller
        )

    time.sleep(0.5)

    accept_button = driver.execute_script("""
        return arguments[0].getRootNode()
            .querySelector('[data-testid="uc-accept-all-button"]');
    """, container)

    if accept_button is None:
        raise RuntimeError("No se encontró el botón de aceptar cookies.")

    driver.execute_script("arguments[0].click();", accept_button)
    print("Cookies aceptadas.")



def select_peninsula_if_needed(driver):
    # Find the Peninsula region button
    buttons = driver.find_elements(
        By.CSS_SELECTOR,
        'button[data-region-code="pen"]'
    )

    if not buttons:
        print("Region selection not required.")
        return

    # Wait for the cookie banner to disappear
    try:
        WebDriverWait(driver, 10).until(
            lambda d: not any(
                element.is_displayed()
                for element in d.find_elements(
                    By.CSS_SELECTOR,
                    "#usercentrics-root"
                )
            )
        )
    except TimeoutException:
        print("Cookie overlay is still present.")

    # Try selecting the Peninsula region
    for button in buttons:
        if button.is_displayed() and button.is_enabled():

            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center'});",
                button
            )

            try:
                WebDriverWait(driver, 5).until(
                    lambda d: button.is_displayed()
                    and button.is_enabled()
                )

                button.click()
                print("Region selected: Peninsula")
                return

            except ElementClickInterceptedException:
                print(
                    "Region button is blocked by the cookie banner."
                )
                raise

    print("Peninsula region button not available.")



def go_to_home(driver):
    driver.get(HOME_URL)
    WebDriverWait(driver, WAIT).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )
    time.sleep(1)
    select_peninsula_if_needed(driver)

In [34]:
def find_search_box(driver):
    selectors = [
        'input[type="search"]',
        'input[placeholder*="Buscar"]',
        'input[aria-label*="Buscar"]',
        'input[name*="search"]',
        'input[id*="search"]',
    ]

    def visible_input(_):
        for selector in selectors:
            for element in driver.find_elements(By.CSS_SELECTOR, selector):
                if element.is_displayed() and element.is_enabled():
                    return element
        return False

    return WebDriverWait(driver, WAIT).until(visible_input)


def search_product(driver, term):
    box = find_search_box(driver)
    box.click()
    box.send_keys(Keys.CONTROL, "a")
    box.send_keys(Keys.BACKSPACE)
    box.send_keys(term)
    box.send_keys(Keys.ENTER)

    WebDriverWait(driver, WAIT).until(
        lambda d: term.lower() in d.find_element(By.TAG_NAME, "body").text.lower()
    )

    WebDriverWait(driver, WAIT).until(
        lambda d: (
            len(d.find_elements(By.CSS_SELECTOR, ".product-tile")) > 0
            or "Productos (0)" in d.find_element(By.TAG_NAME, "body").text
        )
    )
    time.sleep(1)


def get_expected_product_count(driver):
    for tab in driver.find_elements(By.CSS_SELECTOR, '[data-testid="tab"]'):
        text = tab.text.strip()
        if text.startswith("Productos"):
            match = re.search(r"\((\d+)\)", text)
            if match:
                return int(match.group(1))
    return None


def get_loaded_product_count(driver):
    return len(driver.find_elements(By.CSS_SELECTOR, ".product-tile"))


def load_all_products(driver, expected_count, max_clicks=40):
    loaded = get_loaded_product_count(driver)
    clicks = 0
    print(f"Cargados inicialmente: {loaded}/{expected_count}")

    while loaded < expected_count:
        buttons = driver.find_elements(
            By.CSS_SELECTOR,
            '[data-testid="product-tile-grid-load-more-button"]'
        )
        buttons = [
            b for b in buttons
            if b.is_displayed() and b.is_enabled()
        ]

        if not buttons:
            raise RuntimeError(
                f"No aparece 'Mostrar más' con {loaded}/{expected_count} cargados."
            )

        button = buttons[0]
        driver.execute_script(
            "arguments[0].scrollIntoView({block:'center'});",
            button
        )

        previous = loaded

        try:
            button.click()
        except Exception:
            driver.execute_script("arguments[0].click();", button)

        clicks += 1

        WebDriverWait(driver, WAIT).until(
            lambda d: get_loaded_product_count(d) > previous
        )

        loaded = get_loaded_product_count(driver)
        print(f"Click {clicks}: {previous} -> {loaded}/{expected_count}")

        if clicks >= max_clicks and loaded < expected_count:
            raise RuntimeError("Se alcanzó el máximo de clicks.")

    return loaded, clicks

In [35]:
def extract_products(driver, searched_item):
    products = driver.execute_script("""
        function collectProducts(root = document, results = []) {
            const cards = root.querySelectorAll('.product-tile');

            for (const card of cards) {
                const brand = card.querySelector(
                    '.product-tile__content__upper__brand-name'
                );
                const name = card.querySelector(
                    '.product-tile__content__upper__product-name'
                );
                const price = card.querySelector(
                    '[data-testid$="-tag-current-price-amount"]'
                );
                const salesUnit = card.querySelector(
                    '[data-testid$="-tag-sales-unit"]'
                );
                const basePrice = card.querySelector(
                    '[data-testid$="-tag-base-price"]'
                );
                const link = card.querySelector('a.product-tile__action');

                results.push({
                    brand: brand ? brand.innerText.trim() : null,
                    name: name ? name.innerText.trim() : null,
                    price: price ? price.innerText.trim() : null,
                    quantity_raw: salesUnit ? salesUnit.innerText.trim() : null,
                    price_per_unit_raw: basePrice ? basePrice.innerText.trim() : null,
                    url: link ? link.href : null
                });
            }

            for (const el of root.querySelectorAll('*')) {
                if (el.shadowRoot) collectProducts(el.shadowRoot, results);
            }
            return results;
        }
        return collectProducts();
    """)

    df = pd.DataFrame(products)

    if df.empty:
        return df

    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["product_id"] = df["url"].str.extract(
        r"-(\d+)\.html$", expand=False
    )
    df["supermarket"] = "ALDI"
    df["searched_item"] = searched_item
    df["scraped_at"] = datetime.now().isoformat(timespec="seconds")

    return df

In [36]:
driver = start_driver()
time.sleep(2)
accept_cookies(driver)
time.sleep(1)
select_peninsula_if_needed(driver)

all_results = []
summary = []

for i, term in enumerate(TERMS_TO_SCRAPE, start=1):
    success = False
    started = time.monotonic()

    for attempt in range(1, MAX_RETRIES + 2):
        print("\n" + "=" * 70)
        print(f"[{i}/{len(TERMS_TO_SCRAPE)}] {term} · attempt {attempt}")

        try:
            go_to_home(driver)
            search_product(driver, term)

            expected = get_expected_product_count(driver)
            if expected is None:
                raise RuntimeError("No se pudo leer el contador de productos.")

            loaded, clicks = load_all_products(driver, expected)
            df_term = extract_products(driver, term)
            extracted = len(df_term)

            if expected != loaded or expected != extracted:
                raise RuntimeError(
                    f"Esperados {expected}, cargados {loaded}, extraídos {extracted}."
                )

            elapsed = round(time.monotonic() - started, 2)
            all_results.append(df_term)

            summary.append({
                "searched_item": term,
                "expected_products": expected,
                "loaded_products": loaded,
                "extracted_products": extracted,
                "load_more_clicks": clicks,
                "attempts": attempt,
                "status": "OK",
                "seconds": elapsed,
            })

            print("Estado: OK")
            success = True
            break

        except Exception as exc:
            print(f"Intento {attempt} fallido: {type(exc).__name__}: {exc}")
            if attempt < MAX_RETRIES + 1:
                time.sleep(2)

    if not success:
        summary.append({
            "searched_item": term,
            "expected_products": None,
            "loaded_products": None,
            "extracted_products": 0,
            "load_more_clicks": None,
            "attempts": MAX_RETRIES + 1,
            "status": "ERROR",
            "seconds": round(time.monotonic() - started, 2),
        })

df_summary = pd.DataFrame(summary)
display(df_summary)

Cookies aceptadas.
Region selected: Peninsula

[1/1] leche · attempt 1
Region selection not required.
Cargados inicialmente: 9/139
Click 1: 9 -> 18/139
Click 2: 18 -> 27/139
Click 3: 27 -> 36/139
Click 4: 36 -> 45/139
Click 5: 45 -> 54/139
Click 6: 54 -> 63/139
Click 7: 63 -> 72/139
Click 8: 72 -> 81/139
Click 9: 81 -> 90/139
Click 10: 90 -> 99/139
Click 11: 99 -> 108/139
Click 12: 108 -> 117/139
Click 13: 117 -> 126/139
Click 14: 126 -> 135/139
Click 15: 135 -> 139/139
Estado: OK


,searched_item,expected_products,loaded_products,extracted_products,load_more_clicks,attempts,status,seconds
0,leche,139,139,139,15,1,OK,34.33


## RAW — Combine All Search Results into a Single Dataset

In [37]:
# --------------------------------------------------
# PREPARE RAW DATA
# --------------------------------------------------

if not all_results:
    raise RuntimeError("No products were collected.")

df_aldi_raw = pd.concat(
    all_results,
    ignore_index=True
)

print("Raw rows:", len(df_aldi_raw))
print("Unique product IDs:", df_aldi_raw["product_id"].nunique())
print("Raw data ready for processing.")


Raw rows: 139
Unique product IDs: 139
Raw data ready for processing.


In [38]:
driver.quit()
print("Driver cerrado.")

Driver cerrado.


In [39]:
# --------------------------------------------------
# VALIDATE AND SAVE CURRENT AND HISTORICAL RAW DATA
# --------------------------------------------------

import os
import tempfile

if TEST_MODE:
    print("TEST MODE: no files will be saved.")

else:
    # --------------------------------------------------
    # VALIDATE COLLECTED DATA
    # --------------------------------------------------

    if df_aldi_raw.empty:
        raise RuntimeError(
            "No products collected. No files saved."
        )

    if df_summary.empty:
        raise RuntimeError(
            "No search results were recorded. No files saved."
        )

    required_columns = [
        "brand",
        "name",
        "price",
        "quantity_raw",
        "price_per_unit_raw",
        "url",
        "product_id",
        "supermarket",
        "searched_item",
        "scraped_at",
    ]

    missing_columns = [
        column for column in required_columns
        if column not in df_aldi_raw.columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"Missing RAW columns: {missing_columns}"
        )

    # --------------------------------------------------
    # CHECK COLLECTION COMPLETENESS
    # --------------------------------------------------

    all_terms_attempted = (
        len(df_summary) == len(SEARCH_TERMS)
        and set(df_summary["searched_item"]) == set(SEARCH_TERMS)
    )

    successful_searches = df_summary["status"].eq("OK").all()

    collection_complete = bool(
        all_terms_attempted and successful_searches
    )

    # --------------------------------------------------
    # VALIDATE RAW PRODUCT DATA
    # --------------------------------------------------

    numeric_prices = pd.to_numeric(
        df_aldi_raw["price"],
        errors="coerce"
    )

    valid_prices = (
        numeric_prices.notna()
        & numeric_prices.gt(0)
    )

    valid_names = (
        df_aldi_raw["name"].notna()
        & df_aldi_raw["name"].astype(str).str.strip().ne("")
    )

    valid_rows = valid_prices & valid_names

    if not valid_rows.any():
        raise RuntimeError(
            "No products with valid names and prices found. "
            "No files saved."
        )

    invalid_rows = int((~valid_rows).sum())

    if invalid_rows:
        print(
            f"WARNING: {invalid_rows} RAW rows have "
            "missing names or invalid prices."
        )
        print(
            "These rows will be preserved for central "
            "pipeline validation."
        )

    # --------------------------------------------------
    # PROTECT EXISTING HISTORICAL SNAPSHOTS
    # --------------------------------------------------

    if RAW_HISTORY_PATH.exists():
        raise FileExistsError(
            "A RAW historical snapshot already exists "
            f"for {RUN_DATE}: {RAW_HISTORY_PATH}. "
            "No files were modified."
        )

    # --------------------------------------------------
    # PREPARE OUTPUT DIRECTORIES
    # --------------------------------------------------

    RAW_DIR.mkdir(parents=True, exist_ok=True)
    HISTORY_DIR.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------
    # PREPARE TEMPORARY RAW CSV
    # --------------------------------------------------

    def prepare_raw_temp(destination):
        fd, temp_name = tempfile.mkstemp(
            prefix=".aldi_",
            suffix=".tmp",
            dir=destination.parent
        )

        try:
            with os.fdopen(
                fd,
                "w",
                encoding="utf-8-sig",
                newline=""
            ) as file:
                df_aldi_raw.to_csv(
                    file,
                    index=False
                )

            return Path(temp_name)

        except Exception:
            Path(temp_name).unlink(missing_ok=True)
            raise

    history_temp = None
    current_temp = None

    try:
        history_temp = prepare_raw_temp(RAW_HISTORY_PATH)

        if collection_complete:
            current_temp = prepare_raw_temp(RAW_PATH)

        # Save the historical snapshot without overwriting it.
        with history_temp.open("rb") as source:
            with RAW_HISTORY_PATH.open("xb") as target:
                while True:
                    chunk = source.read(1024 * 1024)
                    if not chunk:
                        break
                    target.write(chunk)

        print("Saved historical RAW:", RAW_HISTORY_PATH)

        # Update current RAW only after a successful full run.
        if collection_complete:
            os.replace(current_temp, RAW_PATH)
            print("Saved current RAW:", RAW_PATH)

    finally:
        if history_temp is not None:
            history_temp.unlink(missing_ok=True)

        if current_temp is not None:
            current_temp.unlink(missing_ok=True)

    # --------------------------------------------------
    # COLLECTION SUMMARY
    # --------------------------------------------------

    print("\nCOLLECTION SUMMARY")
    print("Collection date:", RUN_DATE)
    print("RAW rows:", len(df_aldi_raw))

    print(
        "Distinct product IDs:",
        df_aldi_raw["product_id"].nunique()
    )

    print("Searches attempted:", len(df_summary))
    print("Expected searches:", len(SEARCH_TERMS))
    print("Rows requiring validation:", invalid_rows)

    if collection_complete:
        print("\nStatus: COMPLETE")
        print(
            "Current RAW and historical RAW saved."
        )

    else:
        print("\nStatus: PARTIAL")
        print(
            "Historical RAW saved. "
            "Current RAW was not modified."
        )

        failed_searches = df_summary[
            df_summary["status"] != "OK"
        ]

        if not failed_searches.empty:
            display(
                failed_searches[
                    [
                        "searched_item",
                        "status",
                        "expected_products",
                        "loaded_products",
                        "extracted_products",
                        "attempts",
                    ]
                ]
            )

        if not all_terms_attempted:
            print(
                "WARNING: Not all expected search "
                "terms were attempted."
            )

    print("\nSaved outputs:")
    print(RAW_HISTORY_PATH)

    if collection_complete:
        print(RAW_PATH)

TEST MODE: no files will be saved.
